In [ ]:
import os
import sys

sys.stderr = open(os.devnull, "w")

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import psycopg2
import pytz
from datetime import timedelta
from dateutil.relativedelta import relativedelta
from sktime.forecasting.tbats import TBATS
from sklearn.utils import resample
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.layers import Input, Dense, LSTM, Dropout

warnings.filterwarnings('ignore', category=FutureWarning)

tf.random.set_seed(42)
np.random.seed(42)

/Users/jchan/miniconda3/envs/sales_forecasting/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/jchan/miniconda3/envs/sales_forecasting/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/jchan/miniconda3/envs/sales_forecasting/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/jchan/miniconda3/envs/sales_forecasting/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/jchan/miniconda3/envs/sales_forecasting/lib/python3.12/site-packages/

In [2]:
def calculate_prediction_intervals(y_test, forecast, confidence_level=0.95, n_bootstrap=1000):
    """
    Calculate prediction intervals using bootstrapping.

    Parameters:
    - y_test: pandas.Series, actual test values
    - forecast: pandas.Series, forecasted values
    - confidence_level: float, confidence level for prediction interval (e.g., 0.95 for 95%)
    - n_bootstrap: int, number of bootstrap samples

    Returns:
    - lower_bound: pandas.Series, lower bound of prediction interval
    - upper_bound: pandas.Series, upper bound of prediction interval
    """
    # Step 3: Calculate residuals
    residuals = y_test - forecast

    # Step 4: Bootstrapping
    bootstrap_samples = []
    for _ in range(n_bootstrap):
        sample_residuals = np.random.choice(residuals, size=len(residuals), replace=True)
        bootstrap_forecast = forecast + sample_residuals
        bootstrap_samples.append(bootstrap_forecast)

    # Step 6: Calculate prediction intervals
    lower_percentile = (1 - confidence_level) / 2 * 100
    upper_percentile = (1 + confidence_level) / 2 * 100
    forecast_lower = pd.Series(np.percentile(bootstrap_samples, lower_percentile, axis=0), index=y_test.index)
    forecast = pd.Series(np.percentile(bootstrap_samples, 50, axis=0), index=y_test.index)
    forecast_upper = pd.Series(np.percentile(bootstrap_samples, upper_percentile, axis=0), index=y_test.index)

    return forecast_lower, forecast, forecast_upper

In [3]:
# def calculate_oot_gb_prediction_intervals(df, X_oot, oot_gb, target_column='price_total_agg', confidence_level=0.95, n_bootstrap=1000):
#     """
#     Calculate OOT prediction intervals for out-of-time data.

#     Parameters:
#     - df: pandas.DataFrame, training data containing features and target
#     - X_oot: pandas.DataFrame, out-of-time features
#     - oot_gb: OOT model
#     - target_column: str, name of the target column in df (default: 'price_total_agg')
#     - n_bootstrap: int, number of bootstrap samples (default: 1000)
#     - confidence_level: float, confidence level for prediction interval (default: 0.95)

#     Returns:
#     - oot_gb_forecast_lower: pandas.Series, lower bound of prediction interval
#     - oot_gb_forecast: pandas.Series, median forecast
#     - oot_gb_forecast_upper: pandas.Series, upper bound of prediction interval
#     """
#     # Calculate percentiles based on confidence level
#     lower_percentile = (1 - confidence_level) / 2 * 100
#     upper_percentile = (1 + confidence_level) / 2 * 100

#     bootstrap_predictions = []
#     for _ in range(n_bootstrap):
#         # Generate bootstrap sample
#         df_boot = resample(df)
#         # X_boot = scaler.fit_transform(df_boot.drop(columns=[target_column]))
#         # X_boot = np.reshape(X_boot, (X_boot.shape[0], 1, X_boot.shape[1]))
#         X_boot = df_boot.drop(columns=[target_column])
#         y_boot = df_boot[target_column]
        
#         # Train a new model on the bootstrap sample
#         oot_boot_gb = oot_gb
#         oot_boot_gb.fit(X_boot, y_boot)
        
#         # Generate predictions from the bootstrap model
#         # X_oot_scaled = scaler.transform(X_oot)
#         # X_oot_scaled = np.reshape(X_oot_scaled, (X_oot_scaled.shape[0], 1, X_oot_scaled.shape[1]))
#         bootstrap_pred = oot_boot_gb.predict(X_oot)
#         bootstrap_predictions.append(bootstrap_pred)

#     # Calculate lower and upper bounds for the prediction intervals
#     bootstrap_predictions = np.array(bootstrap_predictions)
#     oot_gb_forecast_lower = pd.Series(np.percentile(bootstrap_predictions, lower_percentile, axis=0), index=X_oot.index)
#     oot_gb_forecast = pd.Series(np.percentile(bootstrap_predictions, 50, axis=0), index=X_oot.index)
#     oot_gb_forecast_upper = pd.Series(np.percentile(bootstrap_predictions, upper_percentile, axis=0), index=X_oot.index)

#     return oot_gb_forecast_lower, oot_gb_forecast, oot_gb_forecast_upper

In [4]:
# def calculate_oot_lstm_prediction_intervals(df, X_oot, oot_lstm, target_column='price_total_agg', confidence_level=0.95, n_bootstrap=1000):
#     """
#     Calculate OOT prediction intervals for out-of-time data.

#     Parameters:
#     - df: pandas.DataFrame, training data containing features and target
#     - X_oot: pandas.DataFrame, out-of-time features
#     - oot_lstm: OOT model
#     - target_column: str, name of the target column in df (default: 'price_total_agg')
#     - n_bootstrap: int, number of bootstrap samples (default: 1000)
#     - confidence_level: float, confidence level for prediction interval (default: 0.95)

#     Returns:
#     - oot_lstm_forecast_lower: pandas.Series, lower bound of prediction interval
#     - oot_lstm_forecast: pandas.Series, median forecast
#     - oot_lstm_forecast_upper: pandas.Series, upper bound of prediction interval
#     """
#     # Calculate percentiles based on confidence level
#     lower_percentile = (1 - confidence_level) / 2 * 100
#     upper_percentile = (1 + confidence_level) / 2 * 100

#     bootstrap_predictions = []
#     for _ in range(n_bootstrap):
#         # Generate bootstrap sample
        
#         scaler = StandardScaler()
        
#         df_boot = resample(df)
#         X_boot = scaler.fit_transform(df_boot.drop(columns=[target_column]))
#         X_boot = np.reshape(X_boot, (X_boot.shape[0], 1, X_boot.shape[1]))
#         y_boot = df_boot[target_column]
        
#         # Train a new model on the bootstrap sample
#         oot_boot_lstm = oot_lstm
#         oot_boot_lstm.fit(X_boot, y_boot)
        
#         # Generate predictions from the bootstrap model
#         X_oot_scaled = scaler.transform(X_oot)
#         X_oot_scaled = np.reshape(X_oot_scaled, (X_oot_scaled.shape[0], 1, X_oot_scaled.shape[1]))
#         bootstrap_pred = oot_boot_lstm.predict(X_oot_scaled)
#         bootstrap_predictions.append(bootstrap_pred)

#     # Calculate lower and upper bounds for the prediction intervals
#     bootstrap_predictions = np.array(bootstrap_predictions).squeeze()
#     oot_lstm_forecast_lower = pd.Series(np.percentile(bootstrap_predictions, lower_percentile, axis=0), index=X_oot.index)
#     oot_lstm_forecast = pd.Series(np.percentile(bootstrap_predictions, 50, axis=0), index=X_oot.index)
#     oot_lstm_forecast_upper = pd.Series(np.percentile(bootstrap_predictions, upper_percentile, axis=0), index=X_oot.index)

#     return oot_lstm_forecast_lower, oot_lstm_forecast, oot_lstm_forecast_upper

In [5]:
def calculate_oot_lr_prediction_intervals(df, X_oot, X_oot_gb, X_oot_lstm, target_column='price_total_agg', confidence_level=0.95, n_bootstrap=1000, epochs=1000, batch_size=32, validation_split=0.1):
    """
    Calculate OOT prediction intervals for out-of-time data.

    Parameters:
    - df: pandas.DataFrame, training data containing features and target
    - X_oot: pandas.DataFrame, out-of-time features
    - oot_lr: OOT model
    - target_column: str, name of the target column in df (default: 'price_total_agg')
    - n_bootstrap: int, number of bootstrap samples (default: 1000)
    - confidence_level: float, confidence level for prediction interval (default: 0.95)

    Returns:
    - oot_lr_forecast_lower: pandas.Series, lower bound of prediction interval
    - oot_lr_forecast: pandas.Series, median forecast
    - oot_lr_forecast_upper: pandas.Series, upper bound of prediction interval
    """
    # Calculate percentiles based on confidence level
    lower_percentile = (1 - confidence_level) / 2 * 100
    upper_percentile = (1 + confidence_level) / 2 * 100

    bootstrap_predictions = []
    for _ in range(n_bootstrap):
        # Generate bootstrap sample
        df_boot = resample(df).sort_index(ascending=True)
        df_boot.index = df.index
        X_boot = df_boot.drop(columns=[target_column])
        y_boot = df_boot[target_column]
        
        scaler = StandardScaler()
        X_boot_scaled = scaler.fit_transform(X_boot)
        X_boot_scaled = np.reshape(X_boot_scaled, (X_boot_scaled.shape[0], 1, X_boot_scaled.shape[1]))
        
        # oot_boot_tbats = TBATS(sp=[24, 168], show_warnings=False, n_jobs=-1)
        oot_boot_tbats = TBATS(sp=24, show_warnings=False, n_jobs=-1)
        oot_boot_tbats.fit(y_boot)
        
        gscv = GridSearchCV(GradientBoostingRegressor(random_state=42, n_iter_no_change=3),
                            param_grid={'max_depth': [1, 2, 4, 8, 16],
                                        'subsample': [0.2, 0.4, 0.6, 0.8, 1],
                                        'max_features': ['sqrt', 'log2', None]},
                            scoring='neg_root_mean_squared_error',
                            cv=TimeSeriesSplit(),
                            n_jobs=-1,
                            verbose=1)
        gscv.fit(X_boot, y_boot)
        oot_boot_gb = gscv.best_estimator_
        
        early_stopping = EarlyStopping(patience=3)
        oot_boot_lstm = Sequential()
        oot_boot_lstm.add(Input(shape=(1, 24)))
        oot_boot_lstm.add(LSTM(128, activation='relu', return_sequences=True))
        oot_boot_lstm.add(Dropout(0.5))
        oot_boot_lstm.add(Dense(1))
        oot_boot_lstm.compile(loss='mean_squared_error', optimizer='adam')
        oot_boot_lstm.fit(X_boot_scaled, y_boot, epochs=epochs, batch_size=batch_size, validation_split=validation_split, shuffle=False, callbacks=early_stopping, verbose=2)
        
        oot_tbats_estimations = pd.Series(oot_boot_tbats.predict(fh=X_boot.index), index=X_boot.index)
        oot_gb_estimations = pd.Series(oot_boot_gb.predict(X_boot), index=X_boot.index)
        oot_lstm_estimations = pd.Series(oot_boot_lstm.predict(X_boot_scaled).squeeze(), index=X_boot.index)
        
        oot_estimations = pd.concat([oot_tbats_estimations, oot_gb_estimations, oot_lstm_estimations], axis=1)
        oot_estimations.columns = ['tbats', 'gb', 'lstm']
        
        # Train a new model on the bootstrap sample
        oot_lr = Ridge()
        oot_lr.fit(oot_estimations, y_boot)
        
        # Generate predictions from the bootstrap model
        bootstrap_tbats_pred = oot_boot_tbats.predict(fh=X_oot.index)
        bootstrap_gb_pred = pd.Series(oot_boot_gb.predict(X_oot_gb), index=X_oot.index)
        X_oot_scaled = scaler.transform(X_oot_lstm)
        X_oot_scaled = np.reshape(X_oot_scaled, (X_oot_scaled.shape[0], 1, X_oot_scaled.shape[1]))
        bootstrap_lstm_pred = pd.Series(oot_boot_lstm.predict(X_oot_scaled).squeeze(), index=X_oot.index)

        oot_bootstrap_forecast = pd.concat([bootstrap_tbats_pred, bootstrap_gb_pred, bootstrap_lstm_pred], axis=1)
        oot_bootstrap_forecast.columns = ['tbats', 'gb', 'lstm']
        
        bootstrap_forecast = pd.Series(oot_lr.predict(oot_bootstrap_forecast), index=X_oot.index)
        bootstrap_predictions.append(bootstrap_forecast)

    # Calculate lower and upper bounds for the prediction intervals
    bootstrap_predictions = np.array(bootstrap_predictions)
    oot_lr_forecast_lower = pd.Series(np.percentile(bootstrap_predictions, lower_percentile, axis=0), index=X_oot.index)
    oot_lr_forecast = pd.Series(np.percentile(bootstrap_predictions, 50, axis=0), index=X_oot.index)
    oot_lr_forecast_upper = pd.Series(np.percentile(bootstrap_predictions, upper_percentile, axis=0), index=X_oot.index)

    return oot_lr_forecast_lower, oot_lr_forecast, oot_lr_forecast_upper

In [6]:
sales = pd.read_csv('pedidos-1743765883268.csv', sep=';', encoding='latin1')

In [7]:
df = sales[['account_id', 'sales_channel_id']].value_counts()
df = pd.DataFrame(df).reset_index()

In [8]:
EPOCHS = 1000
BATCH_SIZE = 32
VALID_SPLIT = 0.1

In [9]:
SAO_PAULO_TZ = pytz.timezone('America/Sao_Paulo')
LOOKBACK = 1
SEASONAL_PERIODS = [24, 24*7]
END_DATE = pd.to_datetime(pd.to_datetime(sales['created_date'].max()).strftime("%Y-%m-%d %H:00:00"))
# START_DATE = END_DATE - timedelta(hours=END_DATE.hour)
START_DATE = END_DATE - timedelta(hours=23)
TEST_SIZE = int((END_DATE - START_DATE).seconds / 60**2 + 1)

In [10]:
# OOT_DATE = START_DATE + timedelta(hours=23)

In [11]:
# OOT_PERIODS = int((OOT_DATE - END_DATE).seconds / 60**2 + 1)

In [12]:
# oot_dates = pd.DatetimeIndex([END_DATE+timedelta(hours=h) for h in range(1, OOT_PERIODS)], freq='h')
# df_oot = pd.DataFrame(index=oot_dates)

In [13]:
# df.drop('count', axis=1, inplace=True)
# for account_id in df['account_id'].unique():
#     df = pd.concat([df, pd.DataFrame({'account_id': [account_id], 'sales_channel_id': ['ALL']})], ignore_index=True)

In [14]:
df

,account_id,sales_channel_id,count
0,5f5ed10c-7792-4080-970c-b11398e3f58f,1,3004186
1,9d39683a-47b3-4a18-942d-f1d741a47e8c,1,2542841
2,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,3,1908829
3,5f5ed10c-7792-4080-970c-b11398e3f58f,9,1057418
4,74abffac-6542-4eff-8eae-cc85463a5d02,1,986953
...,...,...,...
272,60ca8452-5b14-481f-a637-8756f527aee8,4,2
273,f9129295-bb08-4330-b60c-9f0beadda521,36,1
274,789ff5c4-21d8-4ed3-ab64-8b46cf2eba9b,2,1
275,457bce7b-9300-4c10-9a97-070b3c0d081d,5,1


In [15]:
df[df['account_id'] == '423a069b-27b8-44ed-9ef3-ca3cf9470970']

,account_id,sales_channel_id,count
67,423a069b-27b8-44ed-9ef3-ca3cf9470970,1,17714
71,423a069b-27b8-44ed-9ef3-ca3cf9470970,2,15934
180,423a069b-27b8-44ed-9ef3-ca3cf9470970,3,586


In [16]:
account_ids = ['f9129295-bb08-4330-b60c-9f0beadda521', '05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '9d39683a-47b3-4a18-942d-f1d741a47e8c']

df_filtered = df[df['account_id'].isin(account_ids)]

In [17]:
df_filtered

,account_id,sales_channel_id,count
1,9d39683a-47b3-4a18-942d-f1d741a47e8c,1,2542841
2,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,3,1908829
5,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,6,876335
7,f9129295-bb08-4330-b60c-9f0beadda521,1,486716
11,f9129295-bb08-4330-b60c-9f0beadda521,2,321190
15,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,5,263987
29,f9129295-bb08-4330-b60c-9f0beadda521,10,106300
48,f9129295-bb08-4330-b60c-9f0beadda521,38,47359
51,f9129295-bb08-4330-b60c-9f0beadda521,16,41229
55,f9129295-bb08-4330-b60c-9f0beadda521,18,29153


In [18]:
dbname = 'railway'
username = 'sinatra'
pwd = '781B3XjpeuqE'
hostname = 'monorail.proxy.rlwy.net'
port = 25096

connection = psycopg2.connect(database=dbname, user=username, password=pwd, host=hostname, port=port)
cursor = connection.cursor()

In [19]:
cursor.execute("select distinct account_id, channel from public.forecast where model='Ensemble_10';")

In [20]:
res_forecast = cursor.fetchall()

In [21]:
len(res_forecast)

0

In [ ]:
id_pairs = list(zip(df_filtered['account_id'], df_filtered['sales_channel_id']))
# id_pairs = list(zip(df['account_id'], df['sales_channel_id']))
summary_dict = {}
forecast_dict = {}
for coverage in [n*10**-1 for n in range(1, 10)]:
    for account_id, sales_channel_id in id_pairs:
        
        if (account_id, str(sales_channel_id)) in res_forecast:
            continue

        print(f'account_id: {account_id}')
        print(f'sales_channel_id: {sales_channel_id}')
        
        summary_dict[(account_id, sales_channel_id)] = {}
        
        if sales_channel_id == 'ALL':
            cond = (sales['account_id'] == account_id) & \
                   (sales['status'].notna())
        else:
            cond = (sales['account_id'] == account_id) & \
                   (sales['sales_channel_id'] == int(sales_channel_id)) & \
                   (sales['status'].notna())
        
        df_client = sales[cond].drop(['account_id', 'sales_channel_id'], axis=1)
        df_client['created_date'] = pd.to_datetime(df_client['created_date'], format='%Y-%m-%d %H:%M:%S.%f %z')
        df_client = df_client.sort_values('created_date').reset_index(drop=True)
        df_client['created_date'] = df_client['created_date'].dt.strftime("%Y-%m-%d %H:00:00").reset_index(drop=True)
        
        df_client_mod = df_client.groupby('created_date').agg(price_total_agg=('price_total', 'sum'), n_orders=('created_date', 'count'))
        df_client_mod.index = pd.to_datetime(df_client_mod.index, format='%Y-%m-%d %H:00:00')
        
        end_date = pd.to_datetime(END_DATE, format='%Y-%m-%d %H:%M:%S')
        start_date = end_date - relativedelta(months=LOOKBACK)
        date_index = pd.Series(pd.date_range(start=start_date, end=end_date, freq='h', name='created_date'))
        df_client_mod = pd.merge(date_index, df_client_mod, how='left', on='created_date').set_index('created_date')
        
        # oot_date_index = pd.Series(df_oot.index, name='created_date')
        # combined_date_index = pd.concat([date_index, oot_date_index], axis=0)
        # df_client_mod = pd.merge(combined_date_index, df_client_mod, how='left', on='created_date').set_index('created_date')
        
        forecast_dict[account_id] = {} if account_id not in forecast_dict else forecast_dict[account_id]
        forecast_dict[account_id][sales_channel_id] = {}
        
        for n, col in enumerate(['price_total_agg', 'n_orders']):
            print(col)
            
            summary_dict[(account_id, sales_channel_id)][f'dataset_{n}'] = {}
            summary_dict[(account_id, sales_channel_id)][f'dataset_{n}']['name'] = col
            
            forecast_dict[account_id][sales_channel_id][f'dataset_{n}'] = {}
            
            df = df_client_mod[col]
            
            df.fillna(0, inplace=True)
            
            lags = [df.shift(l).rename(f'lag_{l}') for l in range(1, 25)]
            df_lag = pd.concat([df, pd.concat(lags, axis=1)], axis=1).dropna().asfreq('h')
            
            # df = df_lag.loc[df_lag.index <= END_DATE].copy()
            df_train = df_lag.loc[df_lag.index < START_DATE].copy()
            df_test = df_lag.loc[(df_lag.index >= START_DATE) & (df_lag.index <= END_DATE)].copy()
            # df_oot = df_lag.loc[df_lag.index > END_DATE].copy()
            
            # X, y = df.iloc[:, 1:], df.iloc[:, 0]
            X_train, y_train = df_train.iloc[:, 1:], df_train.iloc[:, 0]
            X_test, y_test = df_test.iloc[:, 1:], df_test.iloc[:, 0]
            # X_oot = df_oot.iloc[:, 1:]
            
            # tbats = TBATS(sp=[24, 168], show_warnings=False, n_jobs=-1)
            tbats = TBATS(sp=24, show_warnings=False, n_jobs=-1)
            tbats.fit(y_train)
            
            gscv = GridSearchCV(GradientBoostingRegressor(random_state=42, n_iter_no_change=3),
                                param_grid={'max_depth': [1, 2, 4, 8, 16],
                                            'subsample': [0.2, 0.4, 0.6, 0.8, 1],
                                            'max_features': ['sqrt', 'log2', None]},
                                scoring='neg_root_mean_squared_error',
                                cv=TimeSeriesSplit(),
                                n_jobs=-1,
                                verbose=1)
            gscv.fit(X_train, y_train)
            gb = gscv.best_estimator_
            
            tbats_estimations = tbats.predict(fh=X_train.index)
            gb_estimations = pd.Series(gb.predict(X_train), index=X_train.index)
            
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            # X_test_scaled = lstm_scaler.transform(X_test)
            
            X_train_scaled = np.reshape(X_train_scaled, (X_train.shape[0], 1, X_train.shape[1]))
            # X_test_scaled = np.reshape(X_test_scaled, (X_test.shape[0], 1, X_test.shape[1]))
            
            early_stopping = EarlyStopping(patience=3)
            lstm = Sequential()
            lstm.add(Input(shape=(1, 24)))
            lstm.add(LSTM(128, activation='relu', return_sequences=True))
            lstm.add(Dropout(0.5))
            lstm.add(Dense(1))
            lstm.compile(loss='mean_squared_error', optimizer='adam')
            lstm.fit(X_train_scaled, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=VALID_SPLIT, shuffle=False, callbacks=early_stopping, verbose=2)
            
            lstm_estimations = pd.Series(lstm.predict(X_train_scaled).squeeze(), index=X_train.index)
            
            models_estimations = pd.concat([tbats_estimations, gb_estimations, lstm_estimations], axis=1)
            models_estimations.columns = ['tbats', 'gb', 'lstm']
            lr = Ridge()
            lr.fit(models_estimations, y_train)
            estimations = pd.Series(lr.predict(models_estimations), index=X_train.index)
            
            display(pd.DataFrame(lr.coef_, index=models_estimations.columns, columns=['coef']))
            
            train_mae = mean_absolute_error(y_train, estimations)
            train_rmse = root_mean_squared_error(y_train, estimations)
            
            print(f"train RMSE: {train_rmse:.3f}")
            print(f"train MAE:  {train_mae:.3f}")
            
            # tbats_forecast = tbats.predict(fh=X_test.index)
            # tbats_forecast_int = tbats.predict_interval(fh=X_test.index, coverage=coverage)
            # tbats_forecast_lower = tbats_forecast_int.iloc[:, 0]
            # tbats_forecast_upper = tbats_forecast_int.iloc[:, 1]
            
            # gb_forecast = pd.Series(gb.predict(X_test), index=X_test.index)
            # gb_forecast_lower, gb_forecast, gb_forecast_upper = calculate_prediction_intervals(y_test, gb_forecast, confidence_level=coverage, n_bootstrap=300)
            
            # lstm_forecast = pd.Series(lstm.predict(X_test_scaled).squeeze(), index=X_test.index)
            # lstm_forecast_lower, lstm_forecast, lstm_forecast_upper = calculate_prediction_intervals(y_test, lstm_forecast, confidence_level=coverage, n_bootstrap=300)
            
            # models_forecast = pd.concat([tbats_forecast, gb_forecast, lstm_forecast], axis=1)
            # models_forecast.columns = ['tbats', 'gb', 'lstm']
            
            # models_forecast_lower = pd.concat([tbats_forecast_lower, gb_forecast_lower, lstm_forecast_lower], axis=1)
            # models_forecast_upper = pd.concat([tbats_forecast_upper, gb_forecast_upper, lstm_forecast_upper], axis=1)
            
            # forecast = pd.Series(lr.predict(models_forecast), index=X_test.index)
            # forecast_lower, forecast, forecast_upper = calculate_prediction_intervals(y_test, forecast, confidence_level=coverage, n_bootstrap=300)
            
            # forecast = pd.Series(lr.predict(models_forecast), index=X_test.index)
            # forecast_lower = pd.Series(lr.predict(models_forecast_lower), index=X_test.index)
            # forecast_upper = pd.Series(lr.predict(models_forecast_upper), index=X_test.index)
            
            # test_mae = mean_absolute_error(y_test, forecast)
            # test_rmse = root_mean_squared_error(y_test, forecast)

            # print(f"test RMSE: {test_rmse:.3f}")
            # print(f"test MAE:  {test_mae:.3f}")
            
            # forecast_dict[account_id][sales_channel_id][f'dataset_{n}']['forecast'] = forecast
            # forecast_dict[account_id][sales_channel_id][f'dataset_{n}']['forecast_lower'] = forecast_lower
            # forecast_dict[account_id][sales_channel_id][f'dataset_{n}']['forecast_upper'] = forecast_upper
            
            # oot_tbats = TBATS(sp=[24, 168], show_warnings=False, n_jobs=-1)
            # oot_tbats.fit(y)
            
            # # oot_tbats_forecast = oot_tbats.predict(fh=X_oot.index)
            # # oot_tbats_forecast_int = oot_tbats.predict_interval(fh=X_oot.index, coverage=coverage)
            # # oot_tbats_forecast_lower = oot_tbats_forecast_int.iloc[:, 0]
            # # oot_tbats_forecast_upper = oot_tbats_forecast_int.iloc[:, 1]
            
            # oot_gscv = GridSearchCV(GradientBoostingRegressor(loss="quantile", alpha=0.5, random_state=42, n_iter_no_change=3),
            #                         param_grid={'max_depth': [1, 2, 4, 8, 16],
            #                                     'subsample': [0.2, 0.4, 0.6, 0.8, 1],
            #                                     'max_features': ['sqrt', 'log2', None]},
            #                         scoring='neg_mean_absolute_error',
            #                         cv=TimeSeriesSplit(),
            #                         n_jobs=-1,
            #                         verbose=1)        
            # oot_gscv.fit(X, y)
            # oot_gb = oot_gscv.best_estimator_
            
            # scaler = StandardScaler()
            # X_scaled = np.reshape(scaler.fit_transform(X), (X.shape[0], 1, X.shape[1]))
            
            # early_stopping = EarlyStopping(patience=3)
            # oot_lstm = Sequential()
            # oot_lstm.add(Input(shape=(1, 24)))
            # oot_lstm.add(LSTM(128, activation='relu', return_sequences=True))
            # oot_lstm.add(Dropout(0.5))
            # oot_lstm.add(Dense(1))
            # oot_lstm.compile(loss='mean_absolute_error', optimizer='adam')
            # oot_lstm.fit(X_scaled, y, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=VALID_SPLIT, shuffle=False, callbacks=early_stopping, verbose=2)
            
            # oot_tbats_estimations = pd.Series(oot_tbats.predict(fh=X.index), index=X.index)
            # oot_gb_estimations = pd.Series(oot_gb.predict(X), index=X.index)
            # oot_lstm_estimations = pd.Series(oot_lstm.predict(X_scaled).squeeze(), index=X.index)
            
            # oot_estimations = pd.concat([oot_tbats_estimations, oot_gb_estimations, oot_lstm_estimations], axis=1)
            # oot_estimations.columns = ['tbats', 'gb', 'lstm']
            # oot_lr = Ridge()
            # oot_lr.fit(oot_estimations, y)
            
            # display(pd.DataFrame(zip(oot_lr.feature_names_in_, oot_lr.coef_), columns=['feature', 'coef']))
            
            # oot_gb_forecast_ls = []
            # oot_gb_forecast_lower_ls = []
            # oot_gb_forecast_upper_ls = []
            # oot_lstm_forecast_ls = []
            # oot_lstm_forecast_lower_ls = []
            # oot_lstm_forecast_upper_ls = []
            
            # X_oot_gb = X_oot.copy()
            # X_oot_lstm = X_oot.copy()
            # for i in range(len(X_oot)):
            #     x_oot_gb = X_oot_gb.iloc[i].to_frame().T
                
            #     x_oot_lstm = X_oot_lstm.iloc[i].to_frame().T
            #     x_oot_lstm_scaled = scaler.transform(x_oot_lstm)
            #     x_oot_lstm_scaled = np.reshape(x_oot_lstm_scaled, (x_oot_lstm.shape[0], 1, x_oot_lstm.shape[1]))
                
            #     x_oot_gb_forecast = oot_gb.predict(x_oot_gb)[0]
                
            #     x_oot_lstm_forecast = oot_lstm.predict(x_oot_lstm_scaled).squeeze()
                
            #     for j, k in list(zip(range(i+1, len(X_oot)), range(len(X_oot)-i-1))):
            #         X_oot_gb.iloc[j, k] = x_oot_gb_forecast
            #         X_oot_lstm.iloc[j, k] = x_oot_lstm_forecast
                    
            X_oot_gb = X_test.copy()
            X_oot_lstm = X_test.copy()
            # for i in range(len(X_oot)):
            for i in range(len(X_test)):
                x_oot_gb = X_oot_gb.iloc[i].to_frame().T
                
                x_oot_lstm = X_oot_lstm.iloc[i].to_frame().T
                x_oot_lstm_scaled = scaler.transform(x_oot_lstm)
                x_oot_lstm_scaled = np.reshape(x_oot_lstm_scaled, (x_oot_lstm.shape[0], 1, x_oot_lstm.shape[1]))
                
                # x_oot_gb_forecast = oot_gb.predict(x_oot_gb)[0]
                x_oot_gb_forecast = gb.predict(x_oot_gb)[0]
                
                # x_oot_lstm_forecast = oot_lstm.predict(x_oot_lstm_scaled).squeeze()
                x_oot_lstm_forecast = lstm.predict(x_oot_lstm_scaled).squeeze()
                
                # for j, k in list(zip(range(i+1, len(X_oot)), range(len(X_oot)-i-1))):
                for j, k in list(zip(range(i+1, len(X_test)), range(len(X_test)-i-1))):
                    X_oot_gb.iloc[j, k] = x_oot_gb_forecast
                    X_oot_lstm.iloc[j, k] = x_oot_lstm_forecast
            
            # oot_gb_forecast_lower, oot_gb_forecast, oot_gb_forecast_upper = calculate_oot_gb_prediction_intervals(df, X_oot_gb, oot_gb, target_column=col, confidence_level=coverage, n_bootstrap=300)
            # oot_gb_forecast = pd.Series(oot_gb_forecast, index=X_oot.index)
            # oot_gb_forecast_lower = pd.Series(oot_gb_forecast_lower, index=X_oot.index)
            # oot_gb_forecast_upper = pd.Series(oot_gb_forecast_upper, index=X_oot.index)
            
            # oot_lstm_forecast_lower, oot_lstm_forecast, oot_lstm_forecast_upper = calculate_oot_lstm_prediction_intervals(df, X_oot_lstm, oot_lstm, target_column=col, confidence_level=coverage, n_bootstrap=300)
            # oot_lstm_forecast = pd.Series(oot_lstm_forecast, index=X_oot.index)
            # oot_lstm_forecast_lower = pd.Series(oot_lstm_forecast_lower, index=X_oot.index)
            # oot_lstm_forecast_upper = pd.Series(oot_lstm_forecast_upper, index=X_oot.index)
            
            # oot_models_forecast = pd.concat([oot_tbats_forecast, oot_gb_forecast, oot_lstm_forecast], axis=1)
            # oot_models_forecast.columns = ['tbats', 'gb', 'lstm']
            
            # oot_models_forecast_lower = pd.concat([oot_tbats_forecast_lower, oot_gb_forecast_lower, oot_lstm_forecast_lower], axis=1)
            # oot_models_forecast_upper = pd.concat([oot_tbats_forecast_upper, oot_gb_forecast_upper, oot_lstm_forecast_upper], axis=1)
            
            # oot_forecast = pd.Series(oot_lr.predict(oot_models_forecast), index=X_oot.index)
            # oot_forecast_lower = pd.Series(oot_lr.predict(oot_models_forecast_lower), index=X_oot.index)
            # oot_forecast_upper = pd.Series(oot_lr.predict(oot_models_forecast_upper), index=X_oot.index)
            
            # oot_forecast_lower, oot_forecast, oot_forecast_upper = calculate_oot_lr_prediction_intervals(df, X_oot, oot_tbats, oot_gb, oot_lstm, target_column=col, confidence_level=coverage, n_bootstrap=300)
            forecast_lower, forecast, forecast_upper = calculate_oot_lr_prediction_intervals(df_train, X_test, X_oot_gb, X_oot_lstm, target_column=col, confidence_level=coverage, n_bootstrap=10, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=VALID_SPLIT)
            
            test_mae = mean_absolute_error(y_test, forecast)
            test_rmse = root_mean_squared_error(y_test, forecast)

            print(f"test RMSE: {test_rmse:.3f}")
            print(f"test MAE:  {test_mae:.3f}")
            
            # forecast_dict[account_id][sales_channel_id][f'dataset_{n}']['oot_forecast'] = oot_forecast
            # forecast_dict[account_id][sales_channel_id][f'dataset_{n}']['oot_forecast_lower'] = oot_forecast_lower
            # forecast_dict[account_id][sales_channel_id][f'dataset_{n}']['oot_forecast_upper'] = oot_forecast_upper
            
            forecast_dict[account_id][sales_channel_id][f'dataset_{n}']['forecast'] = forecast
            forecast_dict[account_id][sales_channel_id][f'dataset_{n}']['forecast_lower'] = forecast_lower
            forecast_dict[account_id][sales_channel_id][f'dataset_{n}']['forecast_upper'] = forecast_upper
            
            summary_dict[(account_id, sales_channel_id)][f'dataset_{n}']['train RMSE'] = train_rmse
            summary_dict[(account_id, sales_channel_id)][f'dataset_{n}']['test RMSE'] = test_rmse
            summary_dict[(account_id, sales_channel_id)][f'dataset_{n}']['train MAE'] = train_mae
            summary_dict[(account_id, sales_channel_id)][f'dataset_{n}']['test MAE'] = test_mae
            
            plt.figure(figsize=(12, 3))
            sns.lineplot(y_train.iloc[-3*TEST_SIZE:], lw=0.8, label='Train')
            sns.lineplot(estimations.iloc[-3*TEST_SIZE:], lw=0.8, label='Estimation')
            sns.lineplot(y_test, lw=0.8, label='Test', marker='o', markersize=3)
            sns.lineplot(forecast, lw=0.8, label='Forecast')
            sns.lineplot(forecast_lower, c='darkviolet', alpha=0.5, linestyle='--', lw=0.8)
            sns.lineplot(forecast_upper, c='darkviolet', alpha=0.5, linestyle='--', lw=0.8)
            sns.lineplot(2*forecast_lower-forecast, c='darkviolet', alpha=0.3, linestyle='--', lw=0.8)
            sns.lineplot(2*forecast_upper-forecast, c='darkviolet', alpha=0.3, linestyle='--', lw=0.8)
            # sns.lineplot(oot_forecast, lw=0.8, label='OOT Forecast')
            # sns.lineplot(oot_forecast_lower, c='darkviolet', linestyle='--', lw=0.8)
            # sns.lineplot(oot_forecast_upper, c='darkviolet', linestyle='--', lw=0.8)
            plt.fill_between(x=forecast.index,
                             y1=forecast_lower,
                             y2=forecast_upper,
                             color='violet',
                             alpha=0.5)
            plt.fill_between(x=forecast.index,
                             y1=2*forecast_lower-forecast,
                             y2=2*forecast_upper-forecast,
                             color='violet',
                             alpha=0.3)
            # plt.fill_between(x=oot_forecast.index,
            #                 y1=oot_forecast_lower,
            #                 y2=oot_forecast_upper,
            #                 color='violet',
            #                 alpha=0.5)
            plt.legend()
            plt.savefig(f'plots/ensemble/forecast_{account_id}_{sales_channel_id}_{col}_ens.png')
            plt.show();
        
        # with open(f'sql/ensemble/forecast_{account_id}_{sales_channel_id}_ens.sql', 'w') as output_file:
        #     pt_forecast_lower = forecast_dict[account_id][sales_channel_id]['dataset_0']['forecast_lower']
        #     pt_forecast_upper = forecast_dict[account_id][sales_channel_id]['dataset_0']['forecast_upper']
        #     pt_forecast = forecast_dict[account_id][sales_channel_id]['dataset_0']['forecast']
            
        #     no_forecast_lower = forecast_dict[account_id][sales_channel_id]['dataset_1']['forecast_lower']
        #     no_forecast_upper = forecast_dict[account_id][sales_channel_id]['dataset_1']['forecast_upper']
        #     no_forecast = forecast_dict[account_id][sales_channel_id]['dataset_1']['forecast']
        #     for i in range(len(df_test)):
        #         start_time = df_test.index[i]
        #         end_time = start_time + pd.Timedelta(hours=1)
        #         output_file.write(
        #             f"""
        #             INSERT INTO forecast (id, created, modified, platform, store_name, "start", "end",
        #                                   channel, seller, account_id, organization_id, store_id, minutes_interval,
        #                                   model, orders_high, orders_low, orders_mean, sales_high, sales_low, sales_mean)
        #             VALUES (gen_random_uuid(), now(), now(), 1, (select vtexid from vtex_account where id = '{account_id}'::uuid), '{start_time}', '{end_time}',
        #                     {sales_channel_id}, 'ALL', '{account_id}'::uuid, (select organizationid from vtex_account where id = '{account_id}'::uuid), NULL, 60, 'Ensemble',
        #                     {no_forecast_upper.iloc[i]}, {no_forecast_lower.iloc[i]}, {no_forecast.iloc[i]},
        #                     {pt_forecast_upper.iloc[i]}, {pt_forecast_lower.iloc[i]}, {pt_forecast.iloc[i]});
        #             """
        #             )
            
        #     pt_oot_forecast_lower = forecast_dict[account_id][sales_channel_id]['dataset_0']['oot_forecast_lower']
        #     pt_oot_forecast_upper = forecast_dict[account_id][sales_channel_id]['dataset_0']['oot_forecast_upper']
        #     pt_oot_forecast = forecast_dict[account_id][sales_channel_id]['dataset_0']['oot_forecast']
            
        #     no_oot_forecast_lower = forecast_dict[account_id][sales_channel_id]['dataset_1']['oot_forecast_lower']
        #     no_oot_forecast_upper = forecast_dict[account_id][sales_channel_id]['dataset_1']['oot_forecast_upper']
        #     no_oot_forecast = forecast_dict[account_id][sales_channel_id]['dataset_1']['oot_forecast']
        #     for i in range(len(df_oot)):
        #         start_time = df_oot.index[i]
        #         end_time = start_time + pd.Timedelta(hours=1)
        #         output_file.write(
        #             f"""
        #             INSERT INTO forecast (id, created, modified, platform, store_name, "start", "end",
        #                                   channel, seller, account_id, organization_id, store_id, minutes_interval,
        #                                   model, orders_high, orders_low, orders_mean, sales_high, sales_low, sales_mean)
        #             VALUES (gen_random_uuid(), now(), now(), 1, (select vtexid from vtex_account where id = '{account_id}'::uuid), '{start_time}', '{end_time}',
        #                     {sales_channel_id}, 'ALL', '{account_id}'::uuid, (select organizationid from vtex_account where id = '{account_id}'::uuid), NULL, 60, 'Ensemble',
        #                     {no_oot_forecast_upper.iloc[i]}, {no_oot_forecast_lower.iloc[i]}, {no_oot_forecast.iloc[i]},
        #                     {pt_oot_forecast_upper.iloc[i]}, {pt_oot_forecast_lower.iloc[i]}, {pt_oot_forecast.iloc[i]});
        #             """
        #         )
        
        pt_forecast_lower = forecast_dict[account_id][sales_channel_id]['dataset_0']['forecast_lower']
        pt_forecast_upper = forecast_dict[account_id][sales_channel_id]['dataset_0']['forecast_upper']
        pt_forecast = forecast_dict[account_id][sales_channel_id]['dataset_0']['forecast']
        
        no_forecast_lower = forecast_dict[account_id][sales_channel_id]['dataset_1']['forecast_lower']
        no_forecast_upper = forecast_dict[account_id][sales_channel_id]['dataset_1']['forecast_upper']
        no_forecast = forecast_dict[account_id][sales_channel_id]['dataset_1']['forecast']
        for i in range(len(df_test)):
            start_time = df_test.index[i]
            end_time = start_time + pd.Timedelta(hours=1)
            start_time = start_time.tz_localize(SAO_PAULO_TZ)
            end_time = end_time.tz_localize(SAO_PAULO_TZ)
            insert = \
                f"""
                INSERT INTO public.forecast (id, created, modified, platform, store_name, "start", "end",
                                            channel, seller, account_id, organization_id, store_id, minutes_interval,
                                            model, orders_high, orders_low, orders_mean, sales_high, sales_low, sales_mean)
                VALUES (gen_random_uuid(), now(), now(), 1, '{account_id}'::uuid, '{start_time}', '{end_time}',
                        '{sales_channel_id}', 'ALL', '{account_id}'::uuid, '{account_id}'::uuid, NULL, 60, 'Ensemble_{int(100*coverage)}',
                        {no_forecast_upper.iloc[i]}, {no_forecast_lower.iloc[i]}, {no_forecast.iloc[i]},
                        {pt_forecast_upper.iloc[i]}, {pt_forecast_lower.iloc[i]}, {pt_forecast.iloc[i]});
                """
            cursor.execute(insert)
        
        # pt_oot_forecast_lower = forecast_dict[account_id][sales_channel_id]['dataset_0']['oot_forecast_lower']
        # pt_oot_forecast_upper = forecast_dict[account_id][sales_channel_id]['dataset_0']['oot_forecast_upper']
        # pt_oot_forecast = forecast_dict[account_id][sales_channel_id]['dataset_0']['oot_forecast']
        
        # no_oot_forecast_lower = forecast_dict[account_id][sales_channel_id]['dataset_1']['oot_forecast_lower']
        # no_oot_forecast_upper = forecast_dict[account_id][sales_channel_id]['dataset_1']['oot_forecast_upper']
        # no_oot_forecast = forecast_dict[account_id][sales_channel_id]['dataset_1']['oot_forecast']
        # for i in range(len(df_oot)):
        #     start_time = df_oot.index[i]
        #     end_time = start_time + pd.Timedelta(hours=1)
        #     start_time = start_time.tz_localize(SAO_PAULO_TZ)
        #     end_time = end_time.tz_localize(SAO_PAULO_TZ)
        #     oot_insert = \
        #         f"""
        #         INSERT INTO public.forecast (id, created, modified, platform, store_name, "start", "end",
        #                                     channel, seller, account_id, organization_id, store_id, minutes_interval,
        #                                     model, orders_high, orders_low, orders_mean, sales_high, sales_low, sales_mean)
        #         VALUES (gen_random_uuid(), now(), now(), 1, '{account_id}'::uuid, '{start_time}', '{end_time}',
        #                 '{sales_channel_id}', 'ALL', '{account_id}'::uuid, '{account_id}'::uuid, NULL, 60, 'Ensemble_{int(100*coverage)}',
        #                 {no_oot_forecast_upper.iloc[i]}, {no_oot_forecast_lower.iloc[i]}, {no_oot_forecast.iloc[i]},
        #                 {pt_oot_forecast_upper.iloc[i]}, {pt_oot_forecast_lower.iloc[i]}, {pt_oot_forecast.iloc[i]});
        #         """
        #     cursor.execute(oot_insert)
            
        cursor.execute('commit;')

account_id: 9d39683a-47b3-4a18-942d-f1d741a47e8c
sales_channel_id: 1
price_total_agg
Fitting 5 folds for each of 75 candidates, totalling 375 fits
Epoch 1/1000
20/20 - 1s - 32ms/step - loss: 5670629376.0000 - val_loss: 6372186112.0000
Epoch 2/1000
20/20 - 0s - 3ms/step - loss: 5670552064.0000 - val_loss: 6372057088.0000
Epoch 3/1000
20/20 - 0s - 2ms/step - loss: 5670396928.0000 - val_loss: 6371799040.0000
Epoch 4/1000
20/20 - 0s - 2ms/step - loss: 5670116352.0000 - val_loss: 6371355136.0000
Epoch 5/1000
20/20 - 0s - 2ms/step - loss: 5669651968.0000 - val_loss: 6370698752.0000
Epoch 6/1000
20/20 - 0s - 2ms/step - loss: 5669032448.0000 - val_loss: 6369828352.0000
Epoch 7/1000
20/20 - 0s - 2ms/step - loss: 5668165632.0000 - val_loss: 6368743424.0000
Epoch 8/1000
20/20 - 0s - 2ms/step - loss: 5667212800.0000 - val_loss: 6367455232.0000
Epoch 9/1000
20/20 - 0s - 2ms/step - loss: 5665981440.0000 - val_loss: 6365973504.0000
Epoch 10/1000
20/20 - 0s - 2ms/step - loss: 5664688640.0000 - val_los

,coef
tbats,-0.162382
gb,1.608305
lstm,-0.330686


train RMSE: 13328.777
train MAE:  9338.785
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Fitting 5 folds for each of 75 candidates, to

In [ ]:
# cursor.close()

In [ ]:
dataset_0 = pd.DataFrame({k: v['dataset_0'] for k, v in summary_dict.items()}).T
dataset_1 = pd.DataFrame({k: v['dataset_1'] for k, v in summary_dict.items()}).T

In [ ]:
summary = (
    pd
    .concat([dataset_0, dataset_1], axis=0)
    .rename({'name': 'dataset'}, axis=1)
    .sort_index()
    .reset_index(names=['account_id', 'sales_channel_id'])
)
summary.to_csv('summary_ts.csv', sep=';')

summary

# Evaluating Model

In [ ]:
for account_id, sales_channel_id in id_pairs:
    ens_price_forecast = forecast_dict[account_id][sales_channel_id]['dataset_0']['forecast']
    ens_price_forecast_lower = forecast_dict[account_id][sales_channel_id]['dataset_0']['forecast_lower']
    ens_price_forecast_upper = forecast_dict[account_id][sales_channel_id]['dataset_0']['forecast_upper']
    
    forecast_price_ens = pd.concat([y_test, ens_price_forecast, ens_price_forecast_lower, ens_price_forecast_upper], axis=1)
    forecast_price_ens.columns = ['actual', 'forecast', 'lower', 'upper']
    
    df_forecast_price = forecast_price_ens.assign(
        covered_pts=lambda x:
            4*x['actual'].between(x['lower'], x['upper'], inclusive='both') +
            2*(x['actual'].between(2*x['lower']-x['forecast'], x['lower'], inclusive='left') + x['actual'].between(x['upper'], 2*x['upper']-x['forecast'], inclusive='right')) +
            1*(x['actual'].between(3*x['lower']-2*x['forecast'], 2*x['lower']-x['forecast'], inclusive='left') + x['actual'].between(2*x['upper']-x['forecast'], 3*x['upper']-2*x['forecast'], inclusive='right')),
        covered_width=lambda x: x['upper'] - x['lower'],
    )
    
    df_forecast_price_metrics = df_forecast_price.agg(total_covered=('covered_pts', 'sum'), avg_covered=('covered_pts', 'mean'), avg_covered_width=('covered_width', 'mean'))
    
    df_forecast_price_metrics_norm = df_forecast_price_metrics['avg_covered']/np.log(1+df_forecast_price_metrics['avg_covered_width'])
    
    ens_order_forecast = forecast_dict[account_id][sales_channel_id]['dataset_1']['forecast']
    ens_order_forecast_lower = forecast_dict[account_id][sales_channel_id]['dataset_1']['forecast_lower']
    ens_order_forecast_upper = forecast_dict[account_id][sales_channel_id]['dataset_1']['forecast_upper']
    
    forecast_order_ens = pd.concat([y_test, ens_order_forecast, ens_order_forecast_lower, ens_order_forecast_upper], axis=1)
    forecast_order_ens.columns = ['actual', 'forecast', 'lower', 'upper']
    
    df_forecast_order = forecast_order_ens.assign(
        covered_pts=lambda x:
            4*x['actual'].between(x['lower'], x['upper'], inclusive='both') +
            2*(x['actual'].between(2*x['lower']-x['forecast'], x['lower'], inclusive='left') + x['actual'].between(x['upper'], 2*x['upper']-x['forecast'], inclusive='right')) +
            1*(x['actual'].between(3*x['lower']-2*x['forecast'], 2*x['lower']-x['forecast'], inclusive='left') + x['actual'].between(2*x['upper']-x['forecast'], 3*x['upper']-2*x['forecast'], inclusive='right')),
        covered_width=lambda x: x['upper'] - x['lower'],
    )
    
    df_forecast_order_metrics = df_forecast_order.agg(total_covered=('covered_pts', 'sum'), avg_covered=('covered_pts', 'mean'), avg_covered_width=('covered_width', 'mean'))
    
    df_forecast_order_metrics_norm = df_forecast_order_metrics['avg_covered']/np.log(1+df_forecast_order_metrics['avg_covered_width'])
    
    df_forecast_metrics_norm += 0.7*df_forecast_price_metrics_norm + 0.3*df_forecast_order_metrics_norm

In [ ]:
forecast_price_ens = pd.concat([y_test, ens_price_forecast, ens_price_forecast_lower, ens_price_forecast_upper], axis=1)
forecast_price_ens.columns = ['actual', 'forecast', 'lower', 'upper']

In [ ]:
df_forecast_price = forecast_price_ens.assign(
    covered_pts=lambda x:
        4*x['actual'].between(x['lower'], x['upper'], inclusive='both') +
        2*(x['actual'].between(2*x['lower']-x['forecast'], x['lower'], inclusive='left') + x['actual'].between(x['upper'], 2*x['upper']-x['forecast'], inclusive='right')) +
        1*(x['actual'].between(3*x['lower']-2*x['forecast'], 2*x['lower']-x['forecast'], inclusive='left') + x['actual'].between(2*x['upper']-x['forecast'], 3*x['upper']-2*x['forecast'], inclusive='right')),
    covered_width=lambda x: x['upper'] - x['lower'],
)

In [ ]:
df_forecast_price_metrics = df_forecast_price.agg(total_covered=('covered_pts', 'sum'), avg_covered=('covered_pts', 'mean'), avg_covered_width=('covered_width', 'mean'))

In [ ]:
df_forecast_price_metrics_norm = df_forecast_price_metrics['avg_covered']/np.log(1+df_forecast_price_metrics['avg_covered_width'])